# Public Uptime Monitoring Detection

This notebook checks each platform URL from the NIH IC Efforts Landscape spreadsheet for signs of public uptime monitoring:

1. **Status page links** — Scans the homepage HTML for links to known status page services (Statuspage.io, UptimeRobot, Upptime, Cachet, Instatus, etc.) or subdomain patterns like `status.*`
2. **Health endpoints** — Probes common health/status API paths (`/status`, `/health`, `/healthcheck`, `/api/status`, etc.)
3. **HTTP headers** — Checks response headers for monitoring-related indicators

In [ ]:
import openpyxl
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse, urljoin
import re
import time
import warnings
warnings.filterwarnings('ignore')

xlsx_path = r'NIH IC Efforts Landscape 2026.04.22.xlsx'
wb = openpyxl.load_workbook(xlsx_path, data_only=True)
ws = wb['Ecosystems']

headers = [cell.value for cell in next(ws.iter_rows(min_row=1, max_row=1))]
rows = []
for row in ws.iter_rows(min_row=2, max_row=ws.max_row, values_only=True):
    rows.append(list(row))

df = pd.DataFrame(rows, columns=headers)
df = df.dropna(subset=['Name'])

print(f"Loaded {len(df)} platforms to check.")
df[['Name', 'URL']].head(10)

## Detection Functions

In [ ]:
REQUEST_TIMEOUT = 15
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
SESSION = requests.Session()
SESSION.headers.update({'User-Agent': USER_AGENT})

STATUS_PAGE_SERVICES = [
    'statuspage.io', 'status.io', 'uptimerobot.com', 'upptime.js.org',
    'instatus.com', 'cachet', 'betteruptime.com', 'openstatus.dev',
    'hyperping.io', 'freshstatus.io', 'pagerduty.com', 'opsgenie.com',
    'healthchecks.io', 'statuscake.com',
]

STATUS_LINK_PATTERNS = re.compile(
    r'status|uptime|system.health|incident|operational',
    re.IGNORECASE
)

HEALTH_ENDPOINTS = [
    '/status', '/health', '/healthcheck', '/api/status',
    '/api/health', '/api/v1/status', '/_health', '/heartbeat',
    '/ping', '/up',
]

MONITORING_HEADERS = [
    'x-uptime', 'x-health', 'x-statuspage', 'x-monitoring',
    'x-pingback', 'x-powered-by-statuspage',
]


def normalize_url(url):
    url = url.strip()
    if not url.startswith('http'):
        url = 'https://' + url
    return url


def check_status_page_links(url):
    """Scan homepage HTML for links to status pages or monitoring services."""
    findings = []
    try:
        resp = SESSION.get(url, timeout=REQUEST_TIMEOUT, verify=False, allow_redirects=True)
        soup = BeautifulSoup(resp.text, 'html.parser')

        for a_tag in soup.find_all('a', href=True):
            href = a_tag['href'].lower()
            text = a_tag.get_text(strip=True).lower()

            for svc in STATUS_PAGE_SERVICES:
                if svc in href:
                    findings.append(f"Status page link: {a_tag['href']} (service: {svc})")

            if STATUS_LINK_PATTERNS.search(href) or STATUS_LINK_PATTERNS.search(text):
                parsed = urlparse(a_tag['href'])
                if parsed.hostname and 'status' in parsed.hostname:
                    findings.append(f"Status subdomain link: {a_tag['href']}")
                elif 'status' in text or 'uptime' in text:
                    if a_tag['href'] not in [f.split(': ')[1].split(' ')[0] for f in findings]:
                        findings.append(f"Possible status link: {a_tag['href']} (text: '{a_tag.get_text(strip=True)}')")

    except Exception as e:
        findings.append(f"[Error fetching homepage: {type(e).__name__}]")
    return findings


def check_health_endpoints(url):
    """Probe common health/status endpoints."""
    findings = []
    parsed = urlparse(url)
    base = f"{parsed.scheme}://{parsed.netloc}"

    for endpoint in HEALTH_ENDPOINTS:
        try:
            resp = SESSION.get(base + endpoint, timeout=8, verify=False, allow_redirects=True)
            if resp.status_code == 200:
                content_type = resp.headers.get('content-type', '')
                body_preview = resp.text[:200].strip()
                is_json = 'json' in content_type
                has_status_keyword = bool(re.search(
                    r'"status"|"healthy"|"ok"|"alive"|"up"|"operational"',
                    body_preview, re.IGNORECASE
                ))

                if is_json or has_status_keyword:
                    findings.append(f"Health endpoint: {endpoint} (200 OK, {'JSON' if is_json else 'HTML'}) — {body_preview[:80]}")
                elif len(body_preview) < 500:
                    findings.append(f"Possible health endpoint: {endpoint} (200 OK) — {body_preview[:80]}")
        except Exception:
            pass
    return findings


def check_monitoring_headers(url):
    """Check HTTP response headers for monitoring indicators."""
    findings = []
    try:
        resp = SESSION.get(url, timeout=REQUEST_TIMEOUT, verify=False, allow_redirects=True)
        for header_name in MONITORING_HEADERS:
            for resp_header in resp.headers:
                if header_name in resp_header.lower():
                    findings.append(f"Header: {resp_header}: {resp.headers[resp_header]}")
    except Exception as e:
        findings.append(f"[Error checking headers: {type(e).__name__}]")
    return findings


def check_status_subdomain(url):
    """Check if a status.* subdomain exists for the site's domain."""
    findings = []
    parsed = urlparse(url)
    domain = parsed.netloc
    domain = re.sub(r'^(www\.)', '', domain)

    parts = domain.split('.')
    if len(parts) >= 2:
        root_domain = '.'.join(parts[-2:])
        status_url = f"https://status.{root_domain}"
        try:
            resp = SESSION.get(status_url, timeout=10, verify=False, allow_redirects=True)
            if resp.status_code == 200:
                findings.append(f"Status subdomain exists: {status_url} (200 OK)")
        except Exception:
            pass
    return findings


def check_site(name, url):
    """Run all checks on a single site."""
    url = normalize_url(url)
    print(f"  Checking {name}...", end=' ')

    results = {
        'status_page_links': check_status_page_links(url),
        'health_endpoints': check_health_endpoints(url),
        'monitoring_headers': check_monitoring_headers(url),
        'status_subdomain': check_status_subdomain(url),
    }

    total = sum(len(v) for v in results.values() if not any('[Error' in x for x in v))
    print(f"found {total} indicator(s)")
    time.sleep(1)
    return results

print("Detection functions ready.")

## Run Checks on All Platforms

This will take a few minutes as it checks each site with a 1-second delay between platforms.

In [ ]:
print("Scanning all platforms for public uptime monitoring...\n")

all_results = {}
for _, row in df.iterrows():
    name = row['Name']
    url = row['URL']
    if pd.isna(url) or not url.strip():
        print(f"  Skipping {name} (no URL)")
        continue
    all_results[name] = check_site(name, url)

print(f"\nDone. Checked {len(all_results)} platforms.")

## Detailed Results

In [ ]:
for name, results in all_results.items():
    has_findings = any(
        findings for findings in results.values()
        if findings and not all('[Error' in f for f in findings)
    )
    if has_findings:
        print(f"\n{'='*60}")
        print(f"  {name}")
        print(f"{'='*60}")
        for category, findings in results.items():
            if findings and not all('[Error' in f for f in findings):
                label = category.replace('_', ' ').title()
                for f in findings:
                    if not f.startswith('[Error'):
                        print(f"  [{label}] {f}")

print("\n\nPlatforms with NO public uptime indicators detected:")
print("-" * 50)
for name, results in all_results.items():
    has_findings = any(
        findings for findings in results.values()
        if findings and not all('[Error' in f for f in findings)
    )
    if not has_findings:
        print(f"  - {name}")

## Summary Table

In [ ]:
def has_real_findings(findings_list):
    return bool(findings_list) and not all('[Error' in f for f in findings_list)

summary_rows = []
for name, results in all_results.items():
    url = df.loc[df['Name'] == name, 'URL'].values[0]
    summary_rows.append({
        'Name': name,
        'URL': url,
        'Status Page Links': has_real_findings(results['status_page_links']),
        'Health Endpoints': has_real_findings(results['health_endpoints']),
        'Monitoring Headers': has_real_findings(results['monitoring_headers']),
        'Status Subdomain': has_real_findings(results['status_subdomain']),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df['Any Detected'] = (
    summary_df['Status Page Links'] |
    summary_df['Health Endpoints'] |
    summary_df['Monitoring Headers'] |
    summary_df['Status Subdomain']
)

detected_count = summary_df['Any Detected'].sum()
print(f"Public uptime monitoring detected: {detected_count}/{len(summary_df)} platforms\n")

summary_df.style.applymap(
    lambda v: 'background-color: #d4edda' if v is True else
              ('background-color: #f8d7da' if v is False else ''),
    subset=['Status Page Links', 'Health Endpoints', 'Monitoring Headers',
            'Status Subdomain', 'Any Detected']
)

## Export Results

In [ ]:
import os
os.makedirs('results', exist_ok=True)

detail_rows = []
for name, results in all_results.items():
    url = df.loc[df['Name'] == name, 'URL'].values[0]
    for category, findings in results.items():
        for finding in findings:
            if not finding.startswith('[Error'):
                detail_rows.append({
                    'Name': name,
                    'URL': url,
                    'Check Type': category.replace('_', ' ').title(),
                    'Finding': finding,
                })

if detail_rows:
    detail_df = pd.DataFrame(detail_rows)
    detail_df.to_csv('results/uptime_check_details.csv', index=False)
    print(f"Detailed findings saved to results/uptime_check_details.csv ({len(detail_df)} rows)")
else:
    print("No findings to export.")

summary_df.to_csv('results/uptime_check_summary.csv', index=False)
print(f"Summary saved to results/uptime_check_summary.csv ({len(summary_df)} rows)")